#  Exploration de Données — Conduite de Nuit
**Projet IA — Analyseur Intelligent de Scènes de Conduite**

Ce notebook explore le dataset BDD100K filtré sur le scénario **Conduite de Nuit** :
- Conditions nocturnes (`night`) et aube/crépuscule (`dawn/dusk`)
- Focus sur la visibilité réduite et la détection en faible luminosité
- 6 classes cibles : pedestrian, car, truck, bus, traffic sign, traffic light

In [ ]:
# ============================================================
# 0. Imports
# ============================================================
import json
import os
import random
import glob
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

print('✅ Imports OK')

In [ ]:
# ============================================================
# 1. Chargement du JSON BDD100K
# Cherche automatiquement le(s) fichier(s) JSON dans data/labels/
# ============================================================
LABELS_DIR = '../data/labels/train'
IMAGES_DIR = '../data/images/train'

json_files = list(Path(LABELS_DIR).glob('*.json'))
if not json_files:
    raise FileNotFoundError(f'Aucun JSON trouvé dans {LABELS_DIR}')

all_labels = []
for jf in json_files:
    print(f'Chargement : {jf.name}')
    with open(jf, 'r') as f:
        data = json.load(f)
    all_labels.extend(data if isinstance(data, list) else [])

print(f'\n✅ Total images chargées : {len(all_labels)}')

In [ ]:
# ============================================================
# 2. Filtrage Conduite de Nuit
# ============================================================
CLASSES_OF_INTEREST = ['pedestrian', 'car', 'truck', 'bus',
                        'traffic sign', 'traffic light']

conduite_nuit = []
for entry in all_labels:
    attrs = entry.get('attributes', {})
    if attrs.get('timeofday') not in ['night', 'dawn/dusk']:
        continue
    cats = [l['category'] for l in entry.get('labels', [])]
    if not any(c in CLASSES_OF_INTEREST for c in cats):
        continue
    conduite_nuit.append(entry)

print(f'✅ Images Conduite de Nuit : {len(conduite_nuit)} / {len(all_labels)} total')
print(f'   Soit {len(conduite_nuit)/len(all_labels)*100:.1f}% du dataset')

In [ ]:
# ============================================================
# 3. Distribution : nuit vs aube/crépuscule + scène + météo
# ============================================================
time_counts    = Counter(e['attributes'].get('timeofday', 'unknown')  for e in conduite_nuit)
scene_counts   = Counter(e['attributes'].get('scene', 'unknown')      for e in conduite_nuit)
weather_counts = Counter(e['attributes'].get('weather', 'unknown')    for e in conduite_nuit)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Conditions des images — Conduite de Nuit', fontsize=13)

axes[0].pie(time_counts.values(),    labels=time_counts.keys(),
            autopct='%1.1f%%', colors=['#2c3e50', '#f39c12'])
axes[0].set_title('Heure du jour')

axes[1].pie(scene_counts.values(),   labels=scene_counts.keys(),
            autopct='%1.1f%%', colors=plt.cm.Set2.colors)
axes[1].set_title('Type de scène')

axes[2].pie(weather_counts.values(), labels=weather_counts.keys(),
            autopct='%1.1f%%', colors=plt.cm.Pastel1.colors)
axes[2].set_title('Conditions météo')

plt.tight_layout()
os.makedirs('../data/conduite_nuit', exist_ok=True)
plt.savefig('../data/conduite_nuit/distribution_conditions.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 4. Distribution des classes + objets par image
# ============================================================
class_counts      = Counter()
objects_per_image = []

for entry in conduite_nuit:
    cats = [
        l['category'] for l in entry.get('labels', [])
        if l['category'] in CLASSES_OF_INTEREST and 'box2d' in l
    ]
    class_counts.update(cats)
    objects_per_image.append(len(cats))

colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart des classes
classes = list(class_counts.keys())
counts  = list(class_counts.values())
axes[0].bar(classes, counts, color=colors[:len(classes)])
axes[0].set_title('Distribution des classes — Conduite de Nuit', fontsize=12)
axes[0].set_xlabel('Classe')
axes[0].set_ylabel("Nombre d'instances")
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(counts):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=9)

# Histogramme objets par image
axes[1].hist(objects_per_image, bins=20, color='#2c3e50', edgecolor='white')
axes[1].set_title('Objets par image', fontsize=12)
axes[1].set_xlabel('Objets / image')
axes[1].set_ylabel('Fréquence')
axes[1].axvline(np.mean(objects_per_image), color='#e74c3c',
                linestyle='--', label=f'Moyenne : {np.mean(objects_per_image):.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/conduite_nuit/distribution_classes.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 5. Distribution des confiances simulées (conditions nocturnes)
# Montre pourquoi les piétons sont le défi principal la nuit
# ============================================================
np.random.seed(42)
night_conf = {
    'car':           np.random.normal(0.82, 0.08, 500),
    'pedestrian':    np.random.normal(0.58, 0.14, 300),  # ← plus difficile la nuit
    'truck':         np.random.normal(0.78, 0.09, 200),
    'traffic light': np.random.normal(0.88, 0.06, 400),  # ← phares visibles = facile
    'traffic sign':  np.random.normal(0.70, 0.11, 250),
    'bus':           np.random.normal(0.75, 0.10, 150),
}

fig, ax = plt.subplots(figsize=(12, 5))
data_to_plot = [np.clip(v, 0, 1) for v in night_conf.values()]
bp = ax.boxplot(data_to_plot, labels=night_conf.keys(), patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Confiances simulées par classe (conditions nocturnes)', fontsize=12)
ax.set_ylabel('Score de confiance')
ax.set_ylim(0, 1)
ax.axhline(0.30, color='orange', linestyle='--', alpha=0.8, label='Seuil recommandé nuit (0.30)')
ax.axhline(0.50, color='red',    linestyle='--', alpha=0.5, label='Seuil standard (0.50)')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/conduite_nuit/confiance_classes_nuit.png', dpi=150)
plt.show()
print('⚠️ Défi principal : piétons ont une confiance ~0.58 vs ~0.82 pour les voitures.')
print('   → Seuil réduit à 0.30 recommandé pour le scénario nuit.')

In [ ]:
# ============================================================
# 6. Heatmap de position des objets (densité spatiale)
# ============================================================
IMG_W, IMG_H = 1280, 720
heatmap = np.zeros((IMG_H, IMG_W))

for entry in conduite_nuit:
    for label in entry.get('labels', []):
        if label['category'] in CLASSES_OF_INTEREST and 'box2d' in label:
            b  = label['box2d']
            x1 = max(0, int(b['x1']))
            y1 = max(0, int(b['y1']))
            x2 = min(IMG_W, int(b['x2']))
            y2 = min(IMG_H, int(b['y2']))
            if x2 > x1 and y2 > y1:
                heatmap[y1:y2, x1:x2] += 1

plt.figure(figsize=(12, 6))
plt.imshow(heatmap, cmap='inferno', aspect='auto')
plt.colorbar(label='Densité de présence')
plt.title('Heatmap de position des objets — Conduite de Nuit', fontsize=12)
plt.xlabel('X (pixels)')
plt.ylabel('Y (pixels)')
plt.tight_layout()
plt.savefig('../data/conduite_nuit/heatmap_objets_nuit.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 7. Visualisation de 6 images nocturnes annotées
# ============================================================
COLORS_MAP = {
    'pedestrian':    '#e74c3c',
    'car':           '#3498db',
    'truck':         '#2ecc71',
    'bus':           '#f39c12',
    'traffic sign':  '#9b59b6',
    'traffic light': '#1abc9c',
}

# Filtre les entrées avec images disponibles
available = [
    e for e in conduite_nuit
    if os.path.exists(os.path.join(IMAGES_DIR, e['name']))
]

if not available:
    print(f'⚠️ Aucune image trouvée dans {IMAGES_DIR}')
    print('   Vérifiez le chemin IMAGES_DIR.')
else:
    samples = random.sample(available, min(6, len(available)))
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.patch.set_facecolor('#1a1a2e')

    for ax, entry in zip(axes.flat, samples):
        img_path = os.path.join(IMAGES_DIR, entry['name'])
        img = Image.open(img_path)
        ax.imshow(img)
        for label in entry.get('labels', []):
            cat = label['category']
            if cat not in COLORS_MAP or 'box2d' not in label:
                continue
            b    = label['box2d']
            rect = patches.Rectangle(
                (b['x1'], b['y1']), b['x2']-b['x1'], b['y2']-b['y1'],
                linewidth=2, edgecolor=COLORS_MAP[cat], facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(b['x1'], b['y1']-5, cat, color=COLORS_MAP[cat],
                    fontsize=8, fontweight='bold')
        tod = entry['attributes'].get('timeofday', '')
        ax.set_title(f"{entry['name']} [{tod}]", fontsize=7, color='white')
        ax.axis('off')

    # Masquer les axes inutilisés
    for ax in axes.flat[len(samples):]:
        ax.set_visible(False)

    plt.suptitle("Exemples d'images Conduite de Nuit avec annotations",
                 fontsize=13, color='white')
    plt.tight_layout()
    plt.savefig('../data/conduite_nuit/sample_images_annotated.png', dpi=150,
                facecolor=fig.get_facecolor())
    plt.show()

In [ ]:
# ============================================================
# 8. Résumé statistique
# ============================================================
total_objects = sum(class_counts.values())
print('='*55)
print('   RÉSUMÉ — DATASET CONDUITE DE NUIT')
print('='*55)
print(f'  Images analysées       : {len(conduite_nuit)}')
print(f'  Nuit                   : {time_counts.get("night", 0)}')
print(f'  Aube / Crépuscule      : {time_counts.get("dawn/dusk", 0)}')
print(f'  Total objets annotés   : {total_objects}')
print(f'  Moy. objets / image    : {np.mean(objects_per_image):.1f}')
print(f'  Max objets / image     : {max(objects_per_image, default=0)}')
print()
print('  Distribution des classes :')
for cls, cnt in class_counts.most_common():
    pct = cnt / total_objects * 100
    bar = '█' * int(pct / 2)
    print(f'    {cls:<20}: {cnt:>5}  ({pct:5.1f}%)  {bar}')
print('='*55)
print()
print('  ⚠️  Défi principal : piétons peu visibles la nuit')
print('  💡 Solution : seuil de confiance réduit à 0.25–0.35')
print('  💡 Augmentation : hsv_v=0.6 pour simuler éclairage variable')